# 第一课：读懂数据与时间线
目标：解释一行行情代表什么，亲自检查数据，手算收益并画清信号与标签时间线。

配套文件：同目录 `量化研究实习_完整学习流程.md`。请使用现有项目 Python 环境，例如 `nonstationary_financial_ml/.venv/Scripts/python.exe`。

这里读取旧因子缓存进行价格计算练习。固定事后股票池、前复权约定和成交量异常线索必须保留；该练习不提供盈利 Alpha 或正式 PIT 证据。所有代码只读取原项目数据。

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

candidates = [Path.cwd(), *Path.cwd().parents, Path(r'C:\Users\19388\Desktop\项目')]
project_root = next((p for p in candidates if (p / 'Robust_Alpha_Discovery/data/raw/akshare_smoke.parquet').is_file()), None)
if project_root is None:
    raise FileNotFoundError('未找到项目缓存，请把 project_root 设置为项目总目录。')
data_path = project_root / 'Robust_Alpha_Discovery/data/raw/akshare_smoke.parquet'
panel = pd.read_parquet(data_path).sort_values(['asset', 'date']).reset_index(drop=True)
print('数据来源：', data_path)
panel.head()

## 任务 1：在看统计结果之前，先写出你的判断

- 一行代表一个什么对象？唯一键是什么？
- `asset` 为什么不适合当作普通整数？
- `open` 和 `close` 的价格约定在哪里核对？
- `volume` 与 `amount` 的单位依据是什么？
- `tradable=True` 是否等于已经证明当时可以按所报价格成交？

我的回答：


In [ ]:
summary = pd.Series({
    'rows': len(panel),
    'assets': panel['asset'].nunique(),
    'dates': panel['date'].nunique(),
    'start': panel['date'].min(),
    'end': panel['date'].max(),
    'duplicate_date_asset': panel.duplicated(['date', 'asset']).sum(),
})
summary

## 任务 2：亲自补充检查

在下面自己写代码：
1. 查看每列数据类型和缺失数量。
2. 统计每个日期的股票数量。
3. 选取某个日期的完整横截面。
4. 检查是否有非正价格，以及 high/low 与 open/close 的关系。
5. 解释这些检查无法发现哪些历史信息或交易问题。

当前缓存的参考盘点为 29,200 行、40 股、730 个日期、重复键 0；差异应先核实文件与口径。

In [ ]:
# 在这里完成任务 2。


In [ ]:
one_stock = panel.loc[panel['asset'].eq('000001'), ['date', 'asset', 'open', 'close']].copy()
one_stock.head(8)

## 任务 3：先手算，再写代码

用上表相邻两天的收盘价，先计算：

- 一期简单收益：`C_t / C_(t-1) - 1`。
- 一期对数收益：`log(C_t / C_(t-1))`。
- 本项目旧版一期反转：一期对数收益取负。

随后在完整 panel 上按股票分组计算，检查每只股票首日应怎样处理。不要直接对整张表 shift，以免跨股票取值。

我的手算与解释：


In [ ]:
# 在这里完成任务 3。


## 任务 4：用玩具数据理解标签（可作为下一课）

下面只有两只虚构股票和五个明确交易日，只练计算与时间，不作统计推断。

在 t 日收盘后作决策，t+1 开盘进入，t+2 开盘退出。对 t=2023-01-03：A 的简单收益是 12/11-1；B 是 18/19-1。

自己为每行构造入场日期、出场日期、简单收益与对数收益。按当前日终数据契约，标签到退出日收盘后才可用于训练。

然后删去一条中间报价：解释为什么不能直接把下一条报价当成下一个交易日。先用独立交易日历对齐；未知报价留空。

In [ ]:
sessions = pd.to_datetime(['2023-01-03', '2023-01-04', '2023-01-05', '2023-01-06', '2023-01-09'])
toy = pd.DataFrame({
    'asset': ['A'] * 5 + ['B'] * 5,
    'date': list(sessions) * 2,
    'open': [10, 11, 12, 11, 11.55, 20, 19, 18, 18.9, 18],
    'close': [10, 12, 11, 11.55, 12, 20, 18, 18.9, 18, 18.5],
})
toy

In [ ]:
# 在这里完成玩具标签与缺报价练习。


## 第一课结束前，关掉资料回答

- 三个项目分别回答什么问题？
- 今天读取的是哪个版本的数据，股票池有什么限制？
- 按股票滚动与按日期横截面处理有什么不同？
- 为什么预测资格不能根据未来收益是否缺失来决定？
- 旧版对数收益、V3 简单收益与实际原价现金账有什么区别？

记录：今天亲自完成了什么、用了哪些帮助、还有什么不能解释、下次只改哪一个变量。